# LoRA Fine-tuning 공통 스켈레톤

Language·Vision·Speech 중 **하나**를 선택합니다. TODO를 완성하기 전에는 전체 실행이 되지 않습니다. 태스크별 모델·데이터·전처리·학습 설정은 직접 구현합니다.

Colab GPU 런타임에서 작업합니다. 동일한 Test 데이터로 학습 전후를 비교하고, 최소 5개 예측을 기록합니다.

## 1. 과제 설정

모델·데이터셋 이름과 선택 이유를 작성합니다. 예시는 영화 리뷰 감성 분석, 이미지 분류, 음성 명령 분류입니다.

In [ ]:
TASK = ""  # Language / Vision / Speech
MODEL_ID = ""
DATASET_ID = ""
SEED = 42
OUTPUT_DIR = "./assignment_runs"


선택 이유: TODO

## 2. 실행 환경

아래 설치 코드는 공통 라이브러리를 설치하는 기본 예시입니다. 먼저 설치 셀과 import 셀을 실행하여 정상 작동하는지 확인합니다. 버전 충돌이 발생하면 오류 메시지와 공식 문서를 참고하여 설치할 버전을 조정합니다.


In [ ]:
%pip install transformers datasets peft accelerate pandas


In [ ]:
import importlib.metadata
import torch
import pandas as pd
from datasets import load_dataset
from transformers import pipeline, set_seed
from peft import LoraConfig, get_peft_model

set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
for package in ("torch", "transformers", "datasets", "peft", "accelerate"):
    print(package, importlib.metadata.version(package))


## 3. 데이터 로딩과 분할

load_dataset으로 데이터를 불러와 Dataset 또는 DatasetDict를 반환합니다. 분할 이름이 다르면 train / validation / test로 맞춥니다. 제공된 함수가 기존 분할을 유지하고 없는 분할만 생성합니다. 분할이 없으면 80% / 10% / 10%로 나눕니다.

분할 결과의 크기와 클래스 분포를 확인합니다. 같은 화자·원본을 묶어야 하는 데이터에는 무작위 분할 대신 그룹별 분할을 적용합니다.


In [ ]:
from datasets import Dataset, DatasetDict

def prepare_splits(data, seed=SEED):
    splits = DatasetDict({"train": data}) if isinstance(data, Dataset) else DatasetDict(data)
    if "train" not in splits:
        raise ValueError("학습용 분할의 이름을 train으로 지정하세요.")
    if "test" not in splits:
        split = splits["train"].train_test_split(test_size=0.1, seed=seed)
        splits["train"], splits["test"] = split["train"], split["test"]
        validation_ratio = 1 / 9
    else:
        validation_ratio = 0.1
    if "validation" not in splits:
        split = splits["train"].train_test_split(test_size=validation_ratio, seed=seed)
        splits["train"], splits["validation"] = split["train"], split["test"]
    return DatasetDict({name: splits[name] for name in ("train", "validation", "test")})

def load_data():
    """반환: Dataset 또는 train을 포함한 DatasetDict."""
    raise NotImplementedError("load_dataset으로 데이터를 불러오세요.")

raw = prepare_splits(load_data())
print(raw)
print(raw["train"][0])


## 4. 모델과 전처리

Tokenizer / Image processor / Audio feature extractor 등을 선택합니다. 새 Classification head를 쓰면 초기화 사실과 레이블 매핑을 기록합니다. map으로 모델 입력과 labels를 구성합니다.

- Language: 텍스트·레이블 처리. 생성형 방식은 Prompt와 정답 구간의 Loss 처리를 결정합니다.
- Vision: 이미지 크기·정규화 처리
- Speech: Sampling rate·길이 처리

추론에서도 같은 전처리 의미를 유지합니다. Test 데이터에 학습용 무작위 증강을 적용하지 않습니다.

In [ ]:
def load_model_and_processor():
    """반환: (이미 학습된 모델, 저장 가능한 Tokenizer 또는 Processor)."""
    raise NotImplementedError("모델과 Processor를 불러오세요.")

model, processor = load_model_and_processor()

def preprocess(example):
    """입력: 원본 샘플 dict. 반환: 모델 입력과 labels를 포함한 dict."""
    raise NotImplementedError("태스크별 전처리를 구현하세요.")

prepared = raw.map(preprocess)
print(prepared["train"][0])


전처리·레이블 매핑·Classification head 초기화 여부: TODO

## 5. 학습 전 평가

pipeline 추론 결과를 정답과 같은 형식의 레이블 목록으로 변환합니다. 예: 정답이 0/1이라면 예측도 0/1로 변환합니다. 생성형 모델이 유효하지 않은 답을 출력하면 정답과 일치하지 않는 레이블로 처리하고 평가에서 제외하지 않습니다.

아래 두 TODO를 완성하면 Accuracy와 결과 표는 자동 계산됩니다. Test 전체에서 평가하며, 정답을 추론 입력에 포함하지 않습니다. 학습 후에도 같은 데이터 순서·전처리·추론 설정을 사용합니다. 분류 외 태스크는 지표를 변경합니다.


In [ ]:
def get_targets(test_data):
    """반환: Test 순서대로 정답 레이블을 담은 list."""
    raise NotImplementedError("정답 열을 공통 레이블 목록으로 변환하세요.")

def predict_labels(current_model, processor, test_data):
    """반환: Test 순서대로 예측 레이블을 담은 list."""
    # TODO: 전달된 모델로 pipeline을 만들고 출력에서 레이블을 추출합니다.
    raise NotImplementedError("pipeline 추론과 예측 레이블 변환을 구현하세요.")

def accuracy(targets, predictions):
    if len(targets) == 0 or len(targets) != len(predictions):
        raise ValueError("정답과 예측은 길이가 같고 비어 있지 않아야 합니다.")
    return sum(a == b for a, b in zip(targets, predictions)) / len(targets)

def evaluate_model(current_model, processor, test_data):
    targets = get_targets(test_data)
    predictions = predict_labels(current_model, processor, test_data)
    if len(targets) != len(test_data):
        raise ValueError("Test의 모든 샘플에 대해 정답과 예측을 반환하세요.")
    metrics = {"accuracy": accuracy(targets, predictions)}
    table = pd.DataFrame({"sample_id": range(len(targets)), "target": targets,
                          "prediction": predictions})
    return metrics, table

before_metrics, before = evaluate_model(model, processor, raw["test"])
print(before_metrics)
display(before.head())


## 6. LoRA 적용

모델의 모듈 이름을 확인한 뒤 target_modules, r, lora_alpha 등을 설정합니다. task_type은 해당 PEFT 지원 방식에 맞게 선택합니다. Classification head를 학습해야 한다면 modules_to_save 등 적절한 방법으로 포함하고, 저장 대상도 확인합니다.

In [ ]:
def make_lora_config(model):
    """반환: 선택한 모델에 맞는 LoraConfig."""
    raise NotImplementedError("LoRA 설정을 작성하세요.")

lora_config = make_lora_config(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


LoRA 설정 이유 및 실제 학습 대상 모듈: TODO

## 7. 학습

TrainingArguments, Data collator, Trainer를 구성합니다. Train은 학습에, Validation은 설정 선택에 사용합니다. Test는 전달하지 않습니다. Padding·labels 및 Loss가 태스크와 맞는지 확인합니다.

In [ ]:
def make_trainer(model, processor, train_data, validation_data):
    """반환: train() 메서드를 가진 Trainer. OUTPUT_DIR을 저장 경로로 사용합니다."""
    raise NotImplementedError("학습 설정과 Trainer를 구성하세요.")

trainer = make_trainer(model, processor, prepared["train"], prepared["validation"])
trainer.train()


## 8. 학습 후 평가

학습 전 평가와 동일한 함수·Test 데이터를 사용합니다. 성능 향상이 없거나 실패한 사례도 그대로 기록합니다.

In [ ]:
after_metrics, after = evaluate_model(model, processor, raw["test"])
assert len(before) >= 5 and len(after) >= 5, "비교 샘플을 최소 5개 기록하세요."
assert before["sample_id"].is_unique and after["sample_id"].is_unique
assert before[["sample_id", "target"]].reset_index(drop=True).equals(
    after[["sample_id", "target"]].reset_index(drop=True)
), "학습 전후 동일한 샘플·순서·정답을 사용하세요."
comparison = before.rename(columns={"prediction": "before"}).merge(
    after[["sample_id", "prediction"]].rename(columns={"prediction": "after"}),
    on="sample_id", validate="one_to_one"
)
display(pd.DataFrame({"before": before_metrics, "after": after_metrics}))
display(comparison)


## 9. 저장과 분석

Adapter를 저장합니다. 재사용에는 호환되는 Base model과 Processor·설정도 필요합니다. 추가 학습한 Classification head가 있다면 함께 저장되는지 확인합니다.

In [ ]:
from pathlib import Path

adapter_dir = Path(OUTPUT_DIR) / "adapter"
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
model.config.save_pretrained(adapter_dir)
print("Base model:", MODEL_ID)
print("Adapter·Processor·모델 설정 저장 위치:", adapter_dir.resolve())


### 결과 분석

- Base model 이름·Revision과 저장 위치: TODO
- 데이터 규모·분할, 평가 지표와 학습 설정: TODO
- 학습 전후 변화와 실패 사례: TODO
- 새 Classification head 학습 등 결과 해석에 영향을 준 조건: TODO
- 한계와 개선 방향: TODO

코드, 실행 출력, 설명과 분석을 모두 포함한 `.ipynb` 파일 하나를 제출합니다. Adapter, ZIP, CSV 등은 별도로 제출하지 않습니다. 실행 출력이 유지된 상태로 저장합니다.